UNIVERSIDAD DE LAS FUERZAS ARMADAS "ESPE"
## Elaborado por: Camila Bohórquez, Andrés Cárdenas y Deivis Quispe
## Grupo 3
## Fecha: 28-01-2026


In [1]:
import oracledb

conn = oracledb.connect(
    user="hr",
    password="camila2004",
    dsn="localhost:1521/xe"
)

cursor = conn.cursor()


1. Crear un bloque PL/SQL para calcular un bono de acuerdo a lo siguiente:
a. Utilice el comando DEFINE para indicar el código del empleado
b. Si el empleado tiene un salario menor a 5000, otorgue en el bono un valor del
10 del salario del empleado
c. Si el empleado tiene un salario entre a 5001 y 10.000, otorgue en el bono un
valor del 15 del salario del empleado
d. Si el empleado tiene un salario mayor a 10.001, otorgue en el bono un valor del
20 del salario del empleado

In [9]:
import oracledb

cur = conn.cursor()

try:
    cur.execute("""
    CREATE TABLE empleados (
        codigo NUMBER PRIMARY KEY,
        salario NUMBER
    )
    """)
except oracledb.DatabaseError:
    pass


cur.execute("DELETE FROM empleados")  
cur.execute("INSERT INTO empleados (codigo, salario) VALUES (101, 4500)")
cur.execute("INSERT INTO empleados (codigo, salario) VALUES (102, 8000)")
cur.execute("INSERT INTO empleados (codigo, salario) VALUES (103, 12000)")
conn.commit()

emp_code = 101
cur.execute("""
SELECT CASE
         WHEN salario < 5000 THEN salario * 0.10
         WHEN salario BETWEEN 5001 AND 10000 THEN salario * 0.15
         ELSE salario * 0.20
       END AS bono
FROM empleados
WHERE codigo = :emp_code
""", emp_code=emp_code)

bono = cur.fetchone()[0]
print(f"El bono del empleado {emp_code} es: {bono}")

cur.close()
conn.close()


El bono del empleado 101 es: 450


2.-Crear una tabla llamada EMP, con similar estructura que la tabla EMPLOYEES, adicione a
esta tabla una columna llamada STARS tipo VARCHAR2 de longitud 50.

In [2]:
import oracledb

cur = conn.cursor()
plsql_block = """
BEGIN
   -- Si la tabla EMP existe, eliminarla
   BEGIN
      EXECUTE IMMEDIATE 'DROP TABLE EMP';
   EXCEPTION
      WHEN OTHERS THEN
         IF SQLCODE != -942 THEN -- -942 = tabla no existe
            RAISE;
         END IF;
   END;

   -- Crear tabla EMP con la misma estructura que EMPLOYEES
   EXECUTE IMMEDIATE 'CREATE TABLE EMP AS SELECT * FROM EMPLOYEES WHERE 1=0';

   -- Agregar columna STARS tipo VARCHAR2(50)
   EXECUTE IMMEDIATE 'ALTER TABLE EMP ADD (STARS VARCHAR2(50))';
END;
"""

# Ejecutar el bloque
cur.execute(plsql_block)
conn.commit()

print("Tabla EMP creada con columna STARS.")


Tabla EMP creada con columna STARS.


3.-Crear un bloque PL/SQL que permita guardar asteriscos en la columna STARS de la tabla
EMP creada de acuerdo a lo siguiente:
a. Utilice el comando DEFINE para pasar el valor del ID del empleado
b. Inicialice la variable v_asterisk con NULL
c. Añada en la columna STARS del empleado un asterisco por cada $1000 del
salario recibido, ejemplo:


In [4]:
import oracledb

cur = conn.cursor()

emp_id = 100  # cámbialo por un ID que realmente exista en EMP

plsql_block = """
DECLARE
   v_emp_id    NUMBER := :emp_id;
   v_salary    NUMBER;
   v_asterisk  VARCHAR2(50) := NULL;
BEGIN
   BEGIN
      SELECT salary INTO v_salary
      FROM EMP
      WHERE employee_id = v_emp_id;
   EXCEPTION
      WHEN NO_DATA_FOUND THEN
         DBMS_OUTPUT.PUT_LINE('Empleado ' || v_emp_id || ' no existe en EMP.');
         RETURN;
   END;

   FOR i IN 1 .. FLOOR(v_salary/1000) LOOP
      v_asterisk := NVL(v_asterisk, '') || '*';
   END LOOP;

   UPDATE EMP
   SET STARS = v_asterisk
   WHERE employee_id = v_emp_id;

   COMMIT;
END;
"""

cur.execute(plsql_block, emp_id=emp_id)
print(f"Intento de actualización para empleado {emp_id} completado.")


Intento de actualización para empleado 100 completado.
